# 01 — Exploratory Data Analysis

Quick look at dataset statistics before training.

In [ ]:
import os
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

DATA_ROOT = Path("../data")
CLS_ROOT  = DATA_ROOT / "classification_data"
VER_ROOT  = DATA_ROOT / "verification_data"
PAIRS_VAL = DATA_ROOT / "verification_pairs_val.txt"


## 1. Classification Data Statistics

In [ ]:
splits = {"train": CLS_ROOT / "train_data",
           "val":   CLS_ROOT / "val_data",
           "test":  CLS_ROOT / "test_data"}

for split, path in splits.items():
    classes = [d for d in path.iterdir() if d.is_dir()]
    imgs_per_class = [len(list(c.glob("*.jpg"))) for c in classes]
    print(f"{split:5s}: {len(classes)} classes, "
          f"{sum(imgs_per_class)} images, "
          f"avg {np.mean(imgs_per_class):.1f} imgs/class, "
          f"min {min(imgs_per_class)} max {max(imgs_per_class)}")


In [ ]:
train_path = CLS_ROOT / "train_data"
classes = [d for d in train_path.iterdir() if d.is_dir()]
imgs_per_class = [len(list(c.glob("*.jpg"))) for c in classes]

plt.figure(figsize=(10, 4))
plt.hist(imgs_per_class, bins=30, color='steelblue', edgecolor='white')
plt.xlabel("Images per class")
plt.ylabel("# Classes")
plt.title("Training set — images per identity")
plt.tight_layout()
plt.show()


## 2. Sample Face Grid

In [ ]:
sample_classes = random.sample(classes, 5)
fig, axes = plt.subplots(5, 5, figsize=(12, 12))
for row, cls in enumerate(sample_classes):
    imgs = list(cls.glob("*.jpg"))
    sample_imgs = random.sample(imgs, min(5, len(imgs)))
    for col, img_path in enumerate(sample_imgs):
        axes[row, col].imshow(Image.open(img_path))
        axes[row, col].axis("off")
        if col == 0:
            axes[row, col].set_title(cls.name, fontsize=9)
plt.suptitle("5 random identities × 5 images", fontsize=13)
plt.tight_layout()
plt.show()


## 3. Verification Pairs Analysis

In [ ]:
pairs = []
with open(PAIRS_VAL) as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) == 3:
            pairs.append((parts[0], parts[1], int(parts[2])))

labels = [p[2] for p in pairs]
n_pos = sum(labels)
n_neg = len(labels) - n_pos
print(f"Total pairs : {len(pairs)}")
print(f"Same person : {n_pos} ({n_pos/len(pairs):.1%})")
print(f"Different   : {n_neg} ({n_neg/len(pairs):.1%})")

plt.bar(["Same (1)", "Different (0)"], [n_pos, n_neg], color=["seagreen", "tomato"])
plt.title("Verification pair label distribution")
plt.show()


## 4. Image Properties

In [ ]:
sample_paths = random.sample([p[0] for p in pairs[:200]], 20)
sizes = []
for p in sample_paths:
    full = DATA_ROOT / p
    if full.exists():
        img = Image.open(full)
        sizes.append(img.size)  # (W, H)

widths  = [s[0] for s in sizes]
heights = [s[1] for s in sizes]
print(f"Width  — min {min(widths)}, max {max(widths)}, mean {np.mean(widths):.0f}")
print(f"Height — min {min(heights)}, max {max(heights)}, mean {np.mean(heights):.0f}")
